# Floquet multipliers before period doubling — replot

Replots the Floquet figures of `Hill_method_corrected.ipynb` from the data it stores in
`data/hill_orbits/floquet_before_period_doubling_12_10_2026.json`.
Run the sweep and the save cell of `Hill_method_corrected.ipynb` first.

12/10/2026: data with $g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$ (was $J_1(\varepsilon_p)E_J\varphi_a^2\varphi_b$). The period-doubling line is now computed from the data.

16/10/2026: the coarse sweeps are merged with fine sweeps around the period doubling (`floquet_zoom_period_doubling.py`,
`floquet_{before,after}_period_doubling_zoom_16_10_2026.json`), so that the collision of $\mu_3, \mu_4$ on the real axis is resolved.
Fig. 4d/4e: $\mu_1, \mu_2$ in black, $\mu_3, \mu_4$ in light blue ($\mu_3$ dotted in Fig. 4d: the branch that crosses $-1$).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import *

# Larger fonts for these figures only (Plot_setting uses 10 pt; annotations follow font.size)
FONT_SIZE = 13
plt.rcParams.update({
    'font.size': FONT_SIZE,
    'axes.labelsize': FONT_SIZE,
    'axes.titlesize': FONT_SIZE,
    'xtick.labelsize': FONT_SIZE,
    'ytick.labelsize': FONT_SIZE,
    'legend.fontsize': FONT_SIZE,
    'figure.figsize': (width_in, width_in/1.3),   # a bit taller than Plot_setting, so the two-line y label fits
})

In [ ]:
with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/hill_orbits/floquet_before_period_doubling_12_10_2026.json", "r") as f:
    data = json.load(f)

eps_p_list = np.array(data["eps_p"])
g2_MHz = np.array(data["g2_MHz"])
eigvals_array = np.array(data["exponents_real"]) + 1j*np.array(data["exponents_imag"])
Poincare_eig = np.array(data["multipliers_real"]) + 1j*np.array(data["multipliers_imag"])
n_exp = eigvals_array.shape[1]   # 2n genuine exponents
print({k: v for k, v in data.items() if not isinstance(v, list)})

In [ ]:
with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/hill_orbits/floquet_after_period_doubling_12_10_2026.json", "r") as f:
    data_after = json.load(f)

eps_p_list_after = np.array(data_after["eps_p"])
g2_MHz_after = np.array(data_after["g2_MHz"])
eigvals_array_after = np.array(data_after["exponents_real"]) + 1j*np.array(data_after["exponents_imag"])
Poincare_eig_after = np.array(data_after["multipliers_real"]) + 1j*np.array(data_after["multipliers_imag"])
n_exp_after = eigvals_array_after.shape[1]   # 2n genuine exponents
print({k: v for k, v in data_after.items() if not isinstance(v, list)})

In [ ]:
# Fine sweeps around the period doubling, from floquet_zoom_period_doubling.py. The grids above step by about 0.3 MHz,
# wider than the window (about 0.25 MHz) where the critical pair is real. They replace the coarse points in their range.
# From here on, g2_MHz / Poincare_eig (and *_after) are the merged data, sorted by g2, with columns (mu_1, mu_2, mu_3, mu_4).
OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/"


def merge_zoom(g2, mu, path):
    with open(path, "r") as f:
        d = json.load(f)
    g2_z = np.array(d["g2_MHz"])
    mu_z = np.array(d["multipliers_real"]) + 1j*np.array(d["multipliers_imag"])
    keep = (g2 < g2_z.min()) | (g2 > g2_z.max())
    g2_m, mu_m = np.concatenate((g2[keep], g2_z)), np.concatenate((mu[keep], mu_z))
    order = np.argsort(g2_m)
    return g2_m[order], mu_m[order]


def label_mu(mu, black_pair):
    """Order the columns as (mu_1, mu_2, mu_3, mu_4). black_pair(row) gives the indices of mu_1, mu_2.
    In a complex pair the member with Im < 0 comes first; in a real pair the one with the larger |mu|
    (for mu_3: the critical branch)."""
    out = np.full_like(mu, np.nan)
    for k, row in enumerate(mu):
        if np.any(np.isnan(row)):
            continue
        i12 = list(black_pair(row))
        i34 = [i for i in range(4) if i not in i12]
        for (a, b), c in zip((i12, i34), (0, 2)):
            p, q = row[a], row[b]
            if max(abs(p.imag), abs(q.imag)) > 1e-5:   # real multipliers carry |Im| ~ 1e-7 (folded exponents)
                swap = p.imag > q.imag
            else:
                swap = abs(p) < abs(q)
            out[k, c], out[k, c+1] = (q, p) if swap else (p, q)
    return out


g2_MHz, Poincare_eig = merge_zoom(g2_MHz, Poincare_eig, OUT_DIR + "hill_orbits/floquet_before_period_doubling_zoom_16_10_2026.json")
Poincare_eig = label_mu(Poincare_eig, lambda r: np.argsort(-r.real)[:2])              # period-1: mu_1, mu_2 have Re > 0
g2_MHz_after, Poincare_eig_after = merge_zoom(g2_MHz_after, Poincare_eig_after, OUT_DIR + "hill_orbits/floquet_after_period_doubling_zoom_16_10_2026.json")
Poincare_eig_after = label_mu(Poincare_eig_after, lambda r: np.argsort(-np.abs(r.imag))[:2])   # period-2: mu_1, mu_2 have the larger |Im|

BLACK, LIGHT_BLUE = "k", "#56B4E9"

In [ ]:
# period doubling of the period-1 orbit: smallest real multiplier crosses -1 (interpolated), orange line
_okb = ~np.isnan(Poincare_eig[:, 0])
_gb, _mb = g2_MHz[_okb], np.min(np.real(Poincare_eig[_okb]), axis=1)
_kb = np.where(_mb < -1)[0][0]
g2_PD = np.interp(-1, [_mb[_kb], _mb[_kb-1]], [_gb[_kb], _gb[_kb-1]])
print(f"g2_PD = {g2_PD:.2f} MHz")

# Neimark-Sacker instability of the period-2 orbit: largest |mu| crosses 1 (interpolated)
_ok = ~np.isnan(Poincare_eig_after[:, 0])
_order = np.argsort(g2_MHz_after[_ok])
_g, _m = g2_MHz_after[_ok][_order], np.max(np.abs(Poincare_eig_after[_ok][_order]), axis=1)
_k = np.where(_m > 1)[0][0]
g2_NS = np.interp(1, [_m[_k-1], _m[_k]], [_g[_k-1], _g[_k]])
print(f"g2_NS = {g2_NS:.2f} MHz")


def shade(ax, x0, x1, kind):
    """Background of a stability region: 'stable' (green), 'unstable' (red) or 'hatched' (oblique grey strokes)."""
    if kind == "stable":
        ax.axvspan(x0, x1, color=(70/256,149/256,106/256), alpha=0.5, lw=0, zorder=0)
    elif kind == "unstable":
        ax.axvspan(x0, x1, color=(176/256,21/256,21/256), alpha=0.5, lw=0, zorder=0)
    elif kind == "hatched":
        ax.axvspan(x0, x1, facecolor="none", edgecolor="grey", hatch="//", alpha=0.5, lw=0, zorder=0)


def label_pairs(ax, lines, x, g2, mu, offsets):
    """Write mu_1, mu_2 and mu_3, mu_4 next to their curves at g2 = x, in the colour of the curve drawn on top."""
    i = np.nanargmin(np.abs(g2 - x))
    for (a, b), dy in zip(((0, 1), (2, 3)), offsets):
        ax.annotate(rf"$\mu_{a+1}, \mu_{b+1}$", (x, np.real(mu[i, b]) + dy),
                    color=lines[b].get_color(), ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.7, lw=0))

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, ax = plt.subplots()
lines = [ax.plot(g2_MHz, np.abs(Poincare_eig[:, i]))[0] for i in range(n_exp)]
ax.axhline([1], ls="--", color="r")
ax.axvline([g2_PD], ls="--", color=(228/256,86/256,0), lw=3)
x_min, x_max = np.nanmin(g2_MHz), np.nanmax(g2_MHz)
shade(ax, x_min, g2_PD, "stable")
shade(ax, g2_PD, x_max, "hatched")
ax.set_xlim(x_min, x_max)
label_pairs(ax, lines, x_min + 0.25*(g2_PD - x_min), g2_MHz, Poincare_eig, offsets=(-0.2, 0.35))
ax.set_xlabel(r"$g_2/2\pi$"+" (MHz)")
ax.set_ylabel(r"$|\mu_i|$")

# Inset: complex plane with Floquet multipliers and unit circle
axins = inset_axes(ax, width="30%", height="30%", loc="upper right")
for i in range(n_exp):
    axins.plot(np.real(Poincare_eig[:, i]), np.imag(Poincare_eig[:, i]), 'o-', ms=3)

# Unit circle in red
theta = np.linspace(0, 2*np.pi, 100)
axins.plot(np.cos(theta), np.sin(theta), 'r-', lw=2, label='unit circle')

axins.axhline(0, color='k', lw=0.5, alpha=0.3)
axins.axvline(0, color='k', lw=0.5, alpha=0.3)
axins.set_aspect('equal')
axins.set_xlabel(r"Re($\mu$)", fontsize=9)
axins.set_ylabel(r"Im($\mu$)", fontsize=9)
axins.tick_params(labelsize=8)
axins.grid(True, alpha=0.3)

plt.tight_layout()
#plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/floquet_multiplier_before_period_doubling_12_10_2026.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots()
lines = [ax.plot(g2_MHz, np.imag(Poincare_eig[:, i]))[0] for i in range(n_exp)]
ax.axhline([-1], ls="--", color="r")
ax.axvline([g2_PD], ls="--", color=(228/256,86/256,0), lw=3)
x_min, x_max = np.nanmin(g2_MHz), np.nanmax(g2_MHz)
shade(ax, x_min, g2_PD, "stable")
shade(ax, g2_PD, x_max, "hatched")
ax.set_xlim(x_min, x_max)
label_pairs(ax, lines, x_min + 0.25*(g2_PD - x_min), g2_MHz, Poincare_eig, offsets=(-0.2, 0.35))
ax.set_xlabel(r"$g_2/2\pi$"+" (MHz)")
ax.set_ylabel(r"$Im(\mu_i)$")

plt.tight_layout()
#plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/floquet_multiplier_before_period_doubling_12_10_2026.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots()
lines = [ax.plot(g2_MHz_after, np.abs(Poincare_eig_after[:, i]))[0] for i in range(n_exp_after)]
ax.axhline([1], ls="--", color="r")
ax.axvline([g2_PD], ls="--", color=(228/256,86/256,0), lw=3)
x_min, x_max = np.nanmin(g2_MHz_after), np.nanmax(g2_MHz_after)
shade(ax, x_min, g2_PD, "hatched")
shade(ax, g2_PD, g2_NS, "stable")
shade(ax, g2_NS, x_max, "unstable")
ax.set_xlim(x_min, x_max)
label_pairs(ax, lines, g2_NS + 0.5*(x_max - g2_NS)+2, g2_MHz_after, Poincare_eig_after, offsets=(-0.15, 0.18))
ax.set_xlabel(r"$g_2/2\pi$"+" (MHz)")
ax.set_ylabel(r"$|\mu_i|$")

# Inset: complex plane with Floquet multipliers and unit circle
axins = inset_axes(ax, width="30%", height="30%", loc="upper right")
for i in range(n_exp):
    axins.plot(np.real(Poincare_eig_after[:, i]), np.imag(Poincare_eig_after[:, i]), 'o-', ms=3)

# Unit circle in red
theta = np.linspace(0, 2*np.pi, 100)
axins.plot(np.cos(theta), np.sin(theta), 'r-', lw=2, label='unit circle')

axins.axhline(0, color='k', lw=0.5, alpha=0.3)
axins.axvline(0, color='k', lw=0.5, alpha=0.3)
axins.set_aspect('equal')
axins.set_xlabel(r"Re($\mu$)", fontsize=9)
axins.set_ylabel(r"Im($\mu$)", fontsize=9)
axins.tick_params(labelsize=8)
axins.grid(True, alpha=0.3)
plt.tight_layout()
#plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/floquet_multiplier_after_period_doubling_12_10_2026.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots()
lines = [ax.plot(g2_MHz_after, np.imag(Poincare_eig_after[:, i]))[0] for i in range(n_exp_after)]
ax.axhline([1], ls="--", color="r")
ax.axvline([g2_PD], ls="--", color=(228/256,86/256,0), lw=3)
x_min, x_max = np.nanmin(g2_MHz_after), np.nanmax(g2_MHz_after)
shade(ax, x_min, g2_PD, "hatched")
shade(ax, g2_PD, g2_NS, "stable")
shade(ax, g2_NS, x_max, "unstable")
ax.set_xlim(x_min, x_max)
label_pairs(ax, lines, g2_NS + 0.5*(x_max - g2_NS)+2, g2_MHz_after, Poincare_eig_after, offsets=(-0.15, 0.18))
ax.set_xlabel(r"$g_2/2\pi$"+" (MHz)")
ax.set_ylabel(r"$Im(\mu_i)$")

plt.tight_layout()
#plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/floquet_multiplier_after_period_doubling_12_10_2026.pdf")
plt.show()

In [ ]:
# Fig. 4d: |mu| of the period-1 orbit (instability when |mu| crosses 1),
# inset: Re mu_3, mu_4 around the collision on the real axis, then the period doubling (mu_3 = -1)
INSET_SIZE = 9
STYLES = [dict(color=BLACK), dict(color=BLACK),
          dict(color=LIGHT_BLUE, ls=":", lw=2.2),   # mu_3: the branch that goes unstable
          dict(color=LIGHT_BLUE)]
DRAW_ORDER = (0, 1, 3, 2)                           # dotted mu_3 on top of mu_4 where they coincide

fig, ax = plt.subplots()
lines = {i: ax.plot(g2_MHz, np.abs(Poincare_eig[:, i]), **STYLES[i])[0] for i in DRAW_ORDER}
ax.axhline(1, ls="--", color="r")
ax.axvline(g2_PD, ls="--", color=(228/256,86/256,0), lw=3)
x_min, x_max = np.nanmin(g2_MHz), np.nanmax(g2_MHz)
shade(ax, x_min, g2_PD, "stable")
shade(ax, g2_PD, x_max, "hatched")
ax.set_xlim(x_min, x_max)
ax.set_ylim(0.3, 1.3)
ax.legend([lines[0], lines[2], lines[3]], [r"$\mu_1, \mu_2$", r"$\mu_3$", r"$\mu_4$"],
          loc="upper left", ncol=3, fontsize=10, handlelength=1.2, columnspacing=0.6, handletextpad=0.3, borderaxespad=0.3)
ax.set_xlabel(r"$g_2/2\pi$"+" (MHz)")
ax.set_ylabel(r"$|\mu_i|$")

axin = ax.inset_axes([0.22, 0.13, 0.42, 0.36])
for i in DRAW_ORDER:
    axin.plot(g2_MHz, np.real(Poincare_eig[:, i]), **{**STYLES[i], "lw": 1.6 if i == 2 else 1.2})
axin.axhline(-1, ls="--", color="r", lw=1)
axin.axvline(g2_PD, ls="--", color=(228/256,86/256,0), lw=1.5)
axin.set_xlim(74.6, 75.9)
axin.set_ylim(-1.06, -0.85)
axin.set_xticks([75, 75.5])
axin.set_yticks([-1, -0.9])
axin.set_ylabel(r"$\mathrm{Re}\,\mu$", fontsize=INSET_SIZE, labelpad=2)
axin.tick_params(labelsize=INSET_SIZE, pad=1)
axin.set_facecolor("white")
plt.tight_layout()
plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/floquet_multiplier_modulus_before_period_doubling_12_10_2026.pdf")
plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/floquet_multiplier_modulus_before_period_doubling_12_10_2026.svg")
plt.show()

In [ ]:
# Fig. 4e: |mu| of the period-2 orbit (Neimark-Sacker: a complex pair crosses |mu| = 1 and keeps growing),
# inset: the multipliers in the complex plane with the unit circle, from g2_PD to the end of the sweep (arrows: increasing g2)
STYLES_AFTER = [dict(color=BLACK), dict(color=BLACK), dict(color=LIGHT_BLUE), dict(color=LIGHT_BLUE)]
INSET_LIM = 1.6


def add_arrows(ax, x, y, color, fractions=(0.35, 0.75), lim=INSET_LIM):
    """Arrow heads along the curve (x, y), ordered by increasing g2, at fractions of the arc length visible in the inset."""
    vis = (np.abs(x) < lim) & (np.abs(y) < lim)
    x, y = x[vis], y[vis]
    s = np.concatenate(([0], np.cumsum(np.hypot(np.diff(x), np.diff(y)))))
    for f in fractions:
        k = min(np.searchsorted(s, f*s[-1]), len(x) - 2)
        ax.annotate("", xy=(x[k+1], y[k+1]), xytext=(x[k], y[k]),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=0, shrinkA=0, shrinkB=0, mutation_scale=9))


g2_a, mu_a = g2_MHz_after, Poincare_eig_after   # sorted by g2

fig, ax = plt.subplots()
for i in range(n_exp_after):
    ax.plot(g2_a, np.abs(mu_a[:, i]), **STYLES_AFTER[i])
ax.axhline(1, ls="--", color="r")
ax.axvline(g2_PD, ls="--", color=(228/256,86/256,0), lw=3)
x_min, x_max = np.nanmin(g2_a), np.nanmax(g2_a)
shade(ax, x_min, g2_PD, "hatched")
shade(ax, g2_PD, g2_NS, "stable")
shade(ax, g2_NS, x_max, "unstable")
ax.set_xlim(x_min, x_max)
ax.set_ylim(0, 5.5)
ax.set_xlabel(r"$g_2/2\pi$"+" (MHz)")
ax.set_ylabel(r"$|\mu_i|$")

axin = ax.inset_axes([0.2, 0.54, 0.28, 0.43])
th = np.linspace(0, 2*np.pi, 300)
axin.plot(np.cos(th), np.sin(th), ls="--", color="r", lw=1)
sel = g2_a >= g2_PD
for i in range(n_exp_after):
    x, y = np.real(mu_a[sel, i]), np.imag(mu_a[sel, i])
    axin.plot(x, y, lw=1.2, **STYLES_AFTER[i])
    add_arrows(axin, x, y, STYLES_AFTER[i]["color"])
k_NS = np.argmin(np.abs(g2_a - g2_NS))
#axin.plot(np.real(mu_a[k_NS, :2]), np.imag(mu_a[k_NS, :2]), "o", ms=3, mfc="white", mec="k", mew=0.8, zorder=5)
axin.set_xlim(-INSET_LIM, INSET_LIM)
axin.set_ylim(-INSET_LIM, INSET_LIM)
axin.set_aspect("equal")
axin.set_xticks([-1, 0, 1]); axin.set_yticks([-1, 0, 1])
axin.set_xlabel(r"$\mathrm{Re}\,\mu$", fontsize=INSET_SIZE, labelpad=1)
axin.set_ylabel(r"$\mathrm{Im}\,\mu$", fontsize=INSET_SIZE, labelpad=2)
axin.tick_params(labelsize=INSET_SIZE, pad=1)
axin.set_facecolor("white")
plt.tight_layout()
plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/floquet_multiplier_modulus_after_period_doubling_12_10_2026.pdf")
plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/floquet_multiplier_modulus_after_period_doubling_12_10_2026.svg")
plt.show()

### Hamiltonian–Hopf collision with loss (period-2 orbit)

The inset of the figure above shows the full complex plane, where the encounter of the two pairs is not visible: it happens in the angle (frequency), while the pairs separate in modulus (rate).
Here the multipliers are shown as $\ln\mu = \ln|\mu| + i\arg\mu$, after removing the mean $\langle\ln|\mu|\rangle$ over the four multipliers. By Liouville, $\prod_i|\mu_i| = \det M = e^{-2\kappa_b T}$ ($T = 2T_a$, code $\kappa_b$),
so $\langle\ln|\mu|\rangle = -\kappa_b T/2$ is a constant, and the rescaled multipliers $\tilde\mu = \mu\,e^{\kappa_b T/2}$ of the two pairs are mirror images: $|\tilde\mu_{1,2}| = 1/|\tilde\mu_{3,4}|$.
$|\tilde\mu| = 1$ plays the role of the unit circle of a conservative system, and the Neimark–Sacker condition $|\mu| = 1$ becomes $\ln|\tilde\mu| = \kappa_b T/2 \approx 0.075$.

- Without loss, the two pairs would sit on $|\tilde\mu| = 1$, collide at the same angle (Krein collision) and leave the circle perpendicularly as a quartet $\tilde\mu, \bar{\tilde\mu}, 1/\tilde\mu, 1/\bar{\tilde\mu}$.
- With loss, they start slightly off the circle ($|\tilde\mu| \approx 1.02$ and $0.98$ just after the period doubling, where $\mu_3, \mu_4$ leave the real axis), approach in angle, and repel in modulus near the closest approach (an avoided collision).
  The pair pushed outwards crosses $|\mu| = 1$ there: the Neimark–Sacker bifurcation is the dissipative remnant of the Hamiltonian–Hopf collision. The closest approach (printed below) and the Neimark–Sacker point coincide within 0.1 MHz.

Numbers along the curves: $g_2/2\pi$ in MHz. Right: angle and rescaled modulus against $g_2$ (orange: period doubling, black dashed: Neimark–Sacker).

In [ ]:
# Hamiltonian-Hopf collision with loss, period-2 orbit. Liouville fixes the mean of ln|mu_i| over the four multipliers
# (= ln(det M)/4 = -kappa_b T/2), so with ln|mu~| = ln|mu| - <ln|mu|> the two complex pairs are exact mirror images:
# ln|mu~_1| = -ln|mu~_2|. ln|mu~| = 0 is the axis on which a conservative system keeps its multipliers (|mu~| = 1), and
# |mu| = 1 (Neimark-Sacker) is ln|mu~| = -<ln|mu|>. Without loss the two pairs would collide on the axis and leave it
# perpendicularly (Krein collision). With loss they approach in frequency (arg mu) and repel in rate (ln|mu~|): an avoided
# crossing, and the pair pushed outwards crosses |mu| = 1 later.
G2_MAX_HH = 90.0
sel = (g2_MHz_after > g2_PD) & (g2_MHz_after < G2_MAX_HH)
g2_h, mu_h = g2_MHz_after[sel], Poincare_eig_after[sel]
ln_mean = np.mean(np.log(np.abs(mu_h)), axis=1)               # = ln(det M)/4 on each row
y_h = np.log(np.abs(mu_h)) - ln_mean[:, None]                 # ln|mu~|
x_h = np.angle(mu_h)/(2*np.pi)                                # arg mu / 2pi (turns per period 2T_a)
y_NS = -np.mean(ln_mean)                                      # |mu| = 1
print(f"<ln|mu|> = {np.mean(ln_mean):.5f} +- {np.std(ln_mean):.1e} (Liouville: constant), so |mu| = 1 is ln|mu~| = {y_NS:.4f}")
complex_34 = np.abs(mu_h[:, 3].imag) > 1e-5                   # mu_3, mu_4 are real just after the period doubling
# closest approach of the two pairs in the plane of ln mu = ln|mu| + i arg mu
dist = np.where(complex_34, np.abs(np.log(mu_h[:, 1]) - np.log(mu_h[:, 3])), np.nan)
k_min = np.nanargmin(dist)
print(f"closest approach at g2/2pi = {g2_h[k_min]:.2f} MHz (|Delta ln mu| = {dist[k_min]:.3f}); Neimark-Sacker at {g2_NS:.2f} MHz")
k_ns = np.argmin(np.abs(g2_h - g2_NS))
UPPER = ((1, BLACK, r"$\mu_1, \mu_2$"), (3, LIGHT_BLUE, r"$\mu_3, \mu_4$"))   # columns with Im mu > 0

fig, axs = plt.subplot_mosaic([["plane", "arg"], ["plane", "mod"]], figsize=(2*width_in, 1.1*width_in),
                              gridspec_kw=dict(width_ratios=(1.15, 1)))
ax = axs["plane"]
G2_MARKS = (76.0, 78.0, 82.0, 86.0)                            # g2/2pi (MHz) written along the curves
for c, color, lab in UPPER:
    ok = complex_34 if c == 3 else np.ones_like(complex_34)
    ax.plot(x_h[ok, c], y_h[ok, c], color=color, lw=1.2, zorder=2, label=lab)
    for g in G2_MARKS:
        k = np.argmin(np.abs(g2_h - g))
        if ok[k]:
            ax.plot(x_h[k, c], y_h[k, c], "o", ms=3.5, color=color, zorder=3)
            ax.annotate(f"{g:g}", (x_h[k, c], y_h[k, c]), xytext=((6, -6) if c == 1 else (-4, 3)), textcoords="offset points",
                        ha=("left" if c == 1 else "right"), va="center", fontsize=8, color=color)
    add_arrows(ax, x_h[ok, c], y_h[ok, c], color, fractions=(0.5,), lim=np.inf)
    ax.plot(x_h[k_ns, c], y_h[k_ns, c], "*", ms=11, mfc="r", mec="k", mew=0.6, zorder=4, label="NS" if c == 1 else None)
ax.plot(np.zeros((~complex_34).sum()*2), y_h[~complex_34][:, 2:].ravel(), "o", ms=3, mfc="white", mec=LIGHT_BLUE, zorder=3,
        label=r"real $\mu_3, \mu_4$")
ax.axhline(0, color="grey", ls=":", lw=1)
ax.axhline(y_NS, color="r", ls="--", lw=1)
ax.set_xlim(-0.006, 0.17)
ax.text(0.168, y_NS + 0.015, r"$|\mu| = 1$", color="r", fontsize=10, va="bottom", ha="right")
ax.text(0.168, -0.04, r"$|\tilde\mu| = 1$ (no loss)", color="grey", fontsize=10, va="top", ha="right")
ax.grid(False)
ax.set_xlabel(r"$\arg\mu/2\pi$")
ax.set_ylabel(r"$\ln|\tilde\mu| = \ln|\mu| - \langle\ln|\mu|\rangle$")
ax.legend(loc="upper left", fontsize=9, handlelength=1.0, handletextpad=0.4, borderaxespad=0.3, labelspacing=0.3)

for key, val, ylab in (("arg", x_h, r"$\arg\mu/2\pi$"), ("mod", y_h, r"$\ln|\tilde\mu|$")):
    a = axs[key]
    for c, color, lab in UPPER:
        ok = complex_34 if (c == 3 and key == "arg") else np.ones_like(complex_34)
        a.plot(g2_h[ok], val[ok, c], color=color, lw=1.2)
    if key == "mod":
        a.plot(g2_h[~complex_34], y_h[~complex_34, 2], color=LIGHT_BLUE, lw=1.2)   # real mu_3, mu_4 (both members)
        a.axhline(y_NS, color="r", ls="--", lw=1)
        a.axhline(0, color="grey", ls=":", lw=1)
        a.set_xlabel(r"$g_2/2\pi$"+" (MHz)")
    else:
        a.tick_params(labelbottom=False)
    a.axvline(g2_PD, ls="--", color=(228/256,86/256,0), lw=1.5)
    a.axvline(g2_NS, ls="--", color="k", lw=1)
    a.set_xlim(g2_PD - 0.3, G2_MAX_HH)
    a.set_ylabel(ylab)
    a.grid(False)
plt.tight_layout()
plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/floquet_hamiltonian_hopf_19_10_2026.pdf")
plt.show()
